In [ ]:

# USDPEN Volume Forecaster — Phase 1: Data + Walk-Forward (FIXED V2)
import pandas as pd
import numpy as np
import pickle
from datetime import time

df = pd.read_excel('fichinH125.xlsx')
df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
df['Time'] = pd.to_datetime(df['Tiempo'], format='%H:%M:%S').dt.time
r_mask = df['Reg'] == 'R'
print(f"R count: {r_mask.sum()}, R Monto: {df.loc[r_mask, 'Monto'].sum()}")
df = df[~r_mask & (df['Time'] < time(13, 30)) & (df['Time'] >= time(9, 0))]

def get_bin(t):
    m = t.hour * 60 + t.minute
    return (m - 540) // 15
df['bin'] = df['Time'].apply(get_bin)
v_td = df.groupby(['Fecha', 'bin'])['Monto'].sum().unstack(fill_value=0)
p_td = df.groupby(['Fecha', 'bin']).apply(lambda x: (x['Precio'] * x['Monto']).sum() / x['Monto'].sum() if x['Monto'].sum() > 0 else 0)
p_td = p_td.unstack().ffill(axis=1).bfill(axis=1)
V_d = v_td.sum(axis=1)
s_td = v_td.div(V_d, axis=0)
vwap_d = (p_td * s_td).sum(axis=1)
dates = v_td.index
wd = dates.weekday
print(f"Rows: {len(df)}, Sessions: {len(dates)}, Zero-bins: {(v_td == 0).sum().sum()}")
print(f"V(d) Mean/Min/Max: {V_d.mean():.2f}, {V_d.min():.2f}, {V_d.max():.2f}")

def get_group(d_idx, m_id):
    w = wd[d_idx]
    if m_id == 'M3': return [i for i, x in enumerate(wd[:d_idx]) if x == w]
    if m_id == 'M4': return [i for i, x in enumerate(wd[:d_idx]) if (x == 4) == (w == 4)]
    if m_id == 'M5': 
        is_mf = w in
        return [i for i, x in enumerate(wd[:d_idx]) if (x in) == is_mf]
    if m_id == 'M6': return [i for i, x in enumerate(wd[:d_idx]) if (x == 3) == (w == 3)]
    return list(range(d_idx))

res_m, res_s = [], []
for i in range(61, len(dates)):
    for t_idx in range(18):
        v_real = v_td.iloc[i, :t_idx+1].sum()
        v_rem_true = v_td.iloc[i, t_idx+1:].sum()
        for m in ['M0', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6']:
            train_idx = get_group(i, m)
            if not train_idx: continue
            if m == 'M0': v_hat_full = V_d.iloc[train_idx].mean()
            else: 
                valid_wd = [j for j in train_idx if wd[j] == wd[i]]
                v_hat_full = V_d.iloc[valid_wd].mean() if valid_wd else V_d.iloc[train_idx].mean()
            if t_idx == 0:
                v_hat_rem = v_hat_full
            elif m in ['M0', 'M1']:
                v_hat_rem = max(v_hat_full - v_real, 0)
            else:
                mu = s_td.iloc[train_idx].mean(axis=0).values
                v_hat_rem = v_real * (mu[t_idx+1:].sum() / mu[:t_idx+1].sum())
                s_hat = mu[t_idx+1:] / (mu[t_idx+1:].sum() if mu[t_idx+1:].sum() > 0 else 1)
                s_true = s_td.iloc[i, t_idx+1:].values / (v_rem_true if v_rem_true > 0 else 1)
                for b_idx, (sh, st) in enumerate(zip(s_hat, s_true)):
                    res_s.append([dates[i], t_idx, m, t_idx+1+b_idx, sh, st, sh-st])
            res_m.append([dates[i], wd[i], t_idx, m, V_d.iloc[i], v_real, v_rem_true, v_hat_rem, v_hat_rem-v_rem_true, (v_hat_rem-v_rem_true)/v_rem_true if v_rem_true>0 else 0])

e_main = pd.DataFrame(res_m, columns=['session_date', 'weekday', 'T', 'model', 'V_true', 'V_realized', 'V_rem_true', 'V_hat_rem', 'e_rem_usd', 'e_rem_pct'])
e_share = pd.DataFrame(res_s, columns=['session_date', 'T', 'model', 'bin_t', 's_hat', 's_true', 'e_share'])
bin_p = p_td.stack().reset_index()
bin_p.columns = ['session_date', 'bin_t', 'P_vwap']
bin_p = bin_p.merge(vwap_d.rename('VWAP_session'), on='session_date')
e_main.to_pickle('errors_main.pkl')
e_share.to_pickle('errors_share.pkl')
bin_p.to_pickle('bin_prices.pkl')
print(f"Saved: Main {e_main.shape}, Share {e_share.shape}, Price {bin_p.shape}")

In [ ]:
# USDPEN Volume Forecaster — Phase 2: Metrics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

em = pd.read_pickle('errors_main.pkl')
T_SUM =

def get_metrics(df):
    return pd.Series({'MAPE': df['e_rem_pct'].abs().mean(), 'RMSE': np.sqrt((df['e_rem_usd']**2).mean()), 'Bias': df['e_rem_usd'].mean(), 
                      'HR10': (df['e_rem_pct'].abs() < 0.1).mean(), 'HR20': (df['e_rem_pct'].abs() < 0.2).mean(), 'MaxAPE': df['e_rem_pct'].abs().max()})

# Output 1 & 2
mape_wide = em[em['T'].isin(T_SUM)].groupby(['model', 'T'])['e_rem_pct'].apply(lambda x: x.abs().mean()).unstack()
print("MAPE Wide Table:\n", mape_wide)
for t in T_SUM:
    print(f"\nMetrics T={t}:\n", em[em['T'] == t].groupby('model').apply(get_metrics))

# Output 3 & 7 (USD Cones & Scatters)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig2, axes2 = plt.subplots(2, 4, figsize=(20, 10))
cone_data = []
for i, m in enumerate(['M0','M1','M2','M3','M4','M5','M6']):
    ax, ax2 = axes.flatten()[i], axes2.flatten()[i]
    dm = em[em['model'] == m]
    stats_t = dm.groupby('T')['e_rem_usd'].agg([lambda x: np.percentile(x, 5), lambda x: np.percentile(x, 25), 'median', lambda x: np.percentile(x, 75), lambda x: np.percentile(x, 95)])
    ax.fill_between(range(18), stats_t.iloc[:,0], stats_t.iloc[:,4], alpha=0.2)
    ax.fill_between(range(18), stats_t.iloc[:,1], stats_t.iloc[:,3], alpha=0.4)
    ax.plot(range(18), stats_t['median'], 'k')
    ax.axhline(50, ls='--', c='r'); ax.axhline(-50, ls='--', c='r'); ax.set_ylim(-150, 150)
    
    ax2.scatter(dm['T'], dm['e_rem_usd'], alpha=0.3)
    ax2.axhline(50, ls='--', c='r'); ax2.axhline(-50, ls='--', c='r'); ax2.set_ylim(-200, 200)
    
    sub = stats_t[(stats_t.iloc[:,0] > -50) & (stats_t.iloc[:,4] < 50)]
    ax.set_title(f"{m} - First T: {sub.index if not sub.empty else 'N/A'}")
    cone_data.append(stats_t.loc[T_SUM, [stats_t.columns, stats_t.columns]].assign(model=m))
print("\nUSD Cone Table (P5/P95):\n", pd.concat(cone_data))
fig.savefig('cone_usd.png'); fig2.savefig('scatter_usd.png')

# Output 4: Percent Cone
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, m in enumerate(['M0','M1','M2','M3','M4','M5','M6']):
    ax = axes.flatten()[i]
    dm = em[em['model'] == m]
    stats_t = dm.groupby('T')['e_rem_pct'].agg([lambda x: np.percentile(x, 5), lambda x: np.percentile(x, 25), 'median', lambda x: np.percentile(x, 75), lambda x: np.percentile(x, 95)])
    ax.fill_between(range(18), stats_t.iloc[:,0], stats_t.iloc[:,4], alpha=0.2)
    ax.plot(range(18), stats_t['median'], 'k')
    ax.axhline(0.2, ls='--', c='r'); ax.axhline(-0.2, ls='--', c='r'); ax.set_ylim(-1, 1)
fig.savefig('cone_pct.png')

# Output 5: MAPE Decay
plt.figure()
for m in em['model'].unique():
    plt.plot(range(18), em[em['model']==m].groupby('T')['e_rem_pct'].apply(lambda x: x.abs().mean()), label=m)
plt.legend(); plt.savefig('mape_decay.png')

# Output 6: Exceedance
exc = em[em['T'].isin(T_SUM)].groupby(['model', 'T']).apply(lambda x: (x['e_rem_usd'].abs() > 50).mean()).unstack()
print("\nExceedance > 50mm:\n", exc)
print("M2 Failure dates T=6:", em[(em['model']=='M2') & (em['T']==6) & (em['e_rem_usd'].abs() > 50)]['session_date'].tolist())

# Output 8: Pairwise
for t in:
    mods = ['M2','M3','M4','M5','M6']
    mat = np.zeros((5,5))
    for i, m1 in enumerate(mods):
        for j, m2 in enumerate(mods):
            v1 = em[(em['T']==t) & (em['model']==m1)]['e_rem_pct'].abs().values
            v2 = em[(em['T']==t) & (em['model']==m2)]['e_rem_pct'].abs().values
            mat[i,j] = stats.ttest_rel(v1, v2).statistic
    print(f"\nPairwise T={t}:\n", pd.DataFrame(mat, index=mods, columns=mods))

# Output 9 & 10
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, m in enumerate(['M0','M1','M2','M3','M4','M5','M6']):
    dm = em[(em['model']==m) & (em['T']==6)]
    axes.flatten()[i].plot(dm['session_date'], dm['e_rem_pct'])
    axes.flatten()[i].axhline(0.2, ls='--'); axes.flatten()[i].axhline(-0.2, ls='--')
plt.savefig('trajectories_t6.png')

plt.figure()
m2_t6 = em[(em['model']=='M2') & (em['T']==6)].copy()
# V_hat_full was V_real + V_hat_rem in logic
m2_t6['score'] = (m2_t6['V_realized'] + m2_t6['V_hat_rem']) / 500 # proxying train mean as 500 for demo
plt.scatter(m2_t6['session_date'], m2_t6['score'], c=m2_t6['weekday'])
plt.savefig('abnormality_m2.png')

In [ ]:
# USDPEN Volume Forecaster — Phase 3: Shares + P&L
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

es = pd.read_pickle('errors_share.pkl')
bp = pd.read_pickle('bin_prices.pkl')
T_SUM =

# Output 1
l1 = es[es['T'].isin(T_SUM)].groupby(['model', 'T', 'session_date'])['e_share'].apply(lambda x: x.abs().sum()).groupby(['model', 'T']).mean().unstack()
print("L1 Share Error:\n", l1)

# Output 2
def peak_acc(df):
    idx_hat = df.groupby('session_date')['s_hat'].idxmax()
    idx_true = df.groupby('session_date')['s_true'].idxmax()
    b_hat = df.loc[idx_hat, 'bin_t'].values
    b_true = df.loc[idx_true, 'bin_t'].values
    hit = (b_hat == b_true).mean()
    near = (np.abs(b_hat - b_true) <= 1).mean()
    return f"{hit:.1%} / {near:.1%}"
print("\nPeak Accuracy:\n", es[es['T'].isin(T_SUM)].groupby(['model', 'T']).apply(peak_acc).unstack())

# Output 3
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
m2 = es[es['model'] == 'M2']
for i, t in enumerate(T_SUM):
    ax = axes.flatten()[i]
    data = m2[m2['T'] == t].groupby('bin_t')['e_share'].agg(['mean', 'sem'])
    ax.bar(data.index, data['mean'], yerr=data['sem'])
    ax.set_title(f"T={t}")
plt.savefig('share_heatmap_m2.png')

# Output 4 & 5
merged = es.merge(bp, on=['session_date', 'bin_t'])
pnl_res = []
for (m, t, d), group in merged.groupby(['model', 'T', 'session_date']):
    if t == 0: continue
    v_rem_true = group['s_true'].sum() # scaled volume
    if v_rem_true < 0.001: continue
    vwap_err = (group['e_share'] * group['P_vwap']).sum()
    pnl = 100e6 * abs(vwap_err) / group['VWAP_session'].iloc
    pnl_res.append([m, t, d, pnl])

pnl_df = pd.DataFrame(pnl_res, columns=['model', 'T', 'session_date', 'PnL_USD'])
print("\nMean PnL USD:\n", pnl_df.groupby(['model', 'T'])['PnL_USD'].mean().unstack())

plt.figure()
pnl_df[(pnl_df['model']=='M2') & (pnl_df['T']==6)]['PnL_USD'].hist(bins=30)
for val in: plt.axvline(val, color='r', ls='--')
plt.savefig('pnl_hist_m2_t6.png')


=== YIELD TABLE ===
     Mar28  Mar29    Mar31  Mar33    Mar36    Mar46
0  0.04225  0.042  0.04225  0.041  0.04275  0.04325

=== VALUATION @ 2026-04-13 ===
           clean      dirty  accrued   mac_dur   mod_dur  convexity     dv01    carry
Mar28  99.773636  99.920152 0.146516  1.933554  1.855173   5.284985 0.018537 0.011486
Mar29  99.425140 100.039666 0.614525  2.739851  2.629416   9.670751 0.026305 0.011434
Mar31  99.478646  99.625163 0.146516  4.597449  4.411081  24.830879 0.043945 0.011453
Mar33  99.285343  99.427420 0.142077  6.224845  5.979679  44.473023 0.059454 0.011098
Mar36 100.668874 101.341011 0.672137  8.172088  7.837056  77.279247 0.079422 0.011785
Mar46 103.851412 104.457067 0.605655 13.428576 12.871875 224.488390 0.134456 0.012286


0.011487000000002467
